In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import pickle

In [10]:
data=pd.read_csv('Churn_Modelling.csv')

### Drop irrelevent features
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

## Encode categorical variables
label_encoder_gender= LabelEncoder()
data['Gender']= label_encoder_gender.fit_transform(data['Gender'])

onehot_encoder_geo = OneHotEncoder(handle_unknown='ignore')
geo_encoder = onehot_encoder_geo.fit_transform(data[['Geography']])

geo_encoded_df = pd.DataFrame(geo_encoder.toarray(), columns=onehot_encoder_geo.get_feature_names_out(['Geography']))

## Combine one hot encoder columns with the original data
data=pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)

## divide the dataset into independent and dependent features
X = data.drop('Exited', axis=1)
Y = data['Exited']

## Split the data in training and testing sets
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

## Scale these features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test= scaler.transform(X_test)

In [11]:
## Save files in pickle files encoder and scalar
with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('onehot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geo, file)

with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

In [12]:
## Define a function to create model and try different parameters
 
def create_model(neurons=32, layers=1, learning_rate=0.001, dropout_rate=0.0):
    model = Sequential()
    # Explicit Input layer for modern Keras
    model.add(Input(shape=(X_train.shape[1],)))
    
    # First hidden layer
    model.add(Dense(neurons, activation='relu'))
    if dropout_rate > 0.0:
        model.add(Dropout(dropout_rate))
    
    # Additional hidden layers
    for _ in range(layers - 1):
        model.add(Dense(neurons, activation='relu'))
        if dropout_rate > 0.0:
            model.add(Dropout(dropout_rate))
            
    # Binary Classification Output Layer
    model.add(Dense(1, activation='sigmoid'))
    
    # Compile model with customizable optimizer
    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    
    return model

In [13]:
## Create a Keras
model = KerasClassifier(build_fn=create_model, epochs=50, batch_size=10, verbose=0)

In [14]:
# Define hyperparameter grid
param_grid = {
    'model__neurons': [16, 32, 64],
    'model__layers': [1, 2, 3],
    'model__learning_rate': [0.001, 0.01],
    'batch_size': [16, 32],
    'epochs': [20, 50]
}

In [ ]:
# Setup GridSearchCV
grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=3, 
    n_jobs=-1
)

# Fit GridSearch
grid_result = grid.fit(X_train, Y_train)

# Output Best Results
print(f"Best Score: {grid_result.best_score_:.4f}")
print(f"Best Parameters: {grid_result.best_params_}")